[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/warlog7997/rag-playbook/blob/master/learn/04-complex-questions/notebook.ipynb)

# Chapter 4 — Complex Questions

## The Real Story

Our hardest test query for the healthcare Jira assistant was: *"How does the fast-track system work?"*

Every technique we built in Phases 1-6 — hybrid search, reranking, contextual embeddings — **failed this query**. The answer wasn't in one section. It required synthesizing information from the triage protocol, the colour-coded routing system, the transfer-ready criteria, and the escalation procedures. No single chunk contained "the answer."

**Phase 7 — Agentic RAG with a ReAct loop — was the FIRST approach to answer this correctly.** The agent searched 3 times, each with a more targeted query, before finding all the relevant sections. Then it synthesized a coherent answer from the combined context.

This was a pivotal moment. We understood that for domain expert questions, the right approach is an **agent that reasons about what to search for**, not just a pipeline that searches once.

---

## The NovaDash Equivalent

**The hard query:**
> "Compare how Free and Pro plans handle user limits, project limits, and what happens when you exceed those limits."

**Why single-query RAG fails:**
- Requires information from at least 3 files: `billing.md`, `projects.md`, `user-management.md`
- A single query about "user limits" returns chunks from user-management only
- No single query retrieves all three relevant sections

## What You'll Learn
- Multi-query expansion: pool size 5 → 18 unique chunks
- Query decomposition for "compare X and Y" questions
- Agentic ReAct loop with full reasoning trace
- Why agentic was FIRST to answer correctly

## Setup

In [ ]:
!pip install anthropic==0.49.0 chromadb sentence-transformers rank-bm25 keyring -q

In [ ]:
import os
os.environ['PYTHONUTF8'] = '1'

import keyring
# keyring.set_password("Anthropic", "API_KEY", "your-key-here")  # run once
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
LEARN_DIR = REPO_ROOT / 'learn'
sys.path.insert(0, str(LEARN_DIR))

print(f"Repo root: {REPO_ROOT}")
print("Setup complete!")

## The Complex Query

Let's first see how single-query RAG handles this — then we'll fix it.

In [ ]:
from utils import hybrid_search
from collections import Counter

# This is the query that breaks single-pass RAG
COMPLEX_QUERY = (
    "Compare how Free and Pro plans handle user limits and project limits, "
    "and what happens when you exceed those limits."
)

print("=" * 70)
print("  The Problem: Single-Query RAG on a Multi-Part Question")
print("=" * 70)
print(f"\nComplex Query:\n  '{COMPLEX_QUERY}'\n")

# Single-query retrieval: what naive RAG does
single_hits = hybrid_search(COMPLEX_QUERY, top_k=5)
sources_seen = Counter(h['source'] for h in single_hits)

print(f"Single query retrieved {len(single_hits)} chunks from {len(sources_seen)} source(s):")
for i, hit in enumerate(single_hits, 1):
    print(f"  [{i}] {hit['source']} > {hit['section']}")

target_sources = {"billing.md", "projects.md", "user-management.md"}
covered = set(sources_seen.keys()) & target_sources
print(f"\nTarget sources needed: {target_sources}")
print(f"Sources actually covered: {covered}")
print(f"\nProblem: Need all 3 files to answer fully, got {len(covered)}.")

## Approach 1: Multi-Query Expansion

Generate **4 different phrasings** of the same question and run them all. Each query fetches 5 chunks. After deduplication, you have ~15-18 unique chunks covering all aspects.

```
Original: "Compare Free and Pro plan limits"
Expanded to:
  1. "Free plan user and project limits"
  2. "Pro plan features and restrictions"
  3. "What happens when you exceed plan limits"
  4. "Plan comparison user seats projects storage"
```

**Best for:** Broad, multi-faceted questions where you want comprehensive coverage.

In [ ]:
import json
import re
from utils import hybrid_search, ask_claude

QUERY_GEN_SYSTEM = """You are a search query generator. Given a complex user question,
generate 4 focused search queries that together will retrieve all the information needed
to answer the question.

Return ONLY a JSON array of 4 query strings, no explanation:
["query 1", "query 2", "query 3", "query 4"]"""


def generate_query_variants(question: str) -> list:
    """Ask Claude Haiku to generate 4 search query variants for a complex question."""
    response = ask_claude(
        system=QUERY_GEN_SYSTEM,
        user=f"Complex question: {question}",
        model="claude-haiku-4-5-20251001",
    )

    try:
        clean = re.sub(r"```json\s*|\s*```", "", response).strip()
        queries = json.loads(clean)
        if isinstance(queries, list):
            return [str(q) for q in queries[:4]]
    except (json.JSONDecodeError, ValueError):
        queries = re.findall(r'"([^"]{10,})"', response)
        if queries:
            return queries[:4]

    # Hard fallback
    return [
        "Free plan user limits and restrictions",
        "Pro plan features user and project limits",
        "what happens when exceeding plan limits",
        "plan comparison Free Pro features",
    ]


print("=" * 70)
print("  Approach 1: Multi-Query Expansion")
print("=" * 70)

# Step 1: Generate query variants with Claude Haiku
print("\nStep 1: Generating 4 query variants with Claude Haiku...")
queries = generate_query_variants(COMPLEX_QUERY)
print(f"\nGenerated {len(queries)} queries:")
for i, q in enumerate(queries, 1):
    print(f"  [{i}] {q}")

In [ ]:
# Step 2: Run hybrid search for each variant and deduplicate by chunk ID
print("\nStep 2: Running hybrid search for each variant...")

all_hits = {}  # id -> hit (deduplication by chunk ID)

for i, q in enumerate(queries, 1):
    hits = hybrid_search(q, top_k=5, fetch_k=10)
    new_count = 0
    for hit in hits:
        if hit["id"] not in all_hits:
            all_hits[hit["id"]] = hit
            new_count += 1
    print(f"  Query {i}: retrieved {len(hits)} -> {new_count} new unique chunks "
          f"(running total: {len(all_hits)})")

pooled_hits = list(all_hits.values())
print(f"\nFinal pool: {len(pooled_hits)} unique chunks (vs {len(single_hits)} from single query)")

# Step 3: Show coverage improvement
print("\nStep 3: Source coverage in the combined pool:")
source_counts = Counter(h["source"] for h in pooled_hits)
for source, count in sorted(source_counts.items()):
    print(f"  {source}: {count} chunk(s)")

covered_multi = set(source_counts.keys()) & target_sources
print(f"\nTarget sources covered: {covered_multi}")
print(f"Coverage: {len(covered_multi)}/{len(target_sources)} needed sources")

In [ ]:
# Step 4: Generate answer from combined context
print("\nStep 4: Generating answer from combined context...")

# Sort by RRF score if available
pooled_hits.sort(key=lambda h: h.get("rrf_score", h.get("score", 0)), reverse=True)

# Take top 8 most relevant chunks for context
top_hits = pooled_hits[:8]
context = "\n\n---\n\n".join([
    f"Source: {h['source']} | Section: {h['section']}\n{h['text']}"
    for h in top_hits
])

answer = ask_claude(
    system="""You are a helpful NovaDash support assistant. Answer the user's question
using ONLY the provided documentation context. Structure your answer clearly with
headers for each aspect of the question. Be specific and cite the exact limits mentioned
in the docs.""",
    user=f"""Documentation context:
{context}

Question: {COMPLEX_QUERY}""",
    model="claude-haiku-4-5-20251001",
)

print("\n" + "=" * 70)
print("  Multi-Query Answer")
print("=" * 70)
print(answer)

## Approach 2: Query Decomposition

Instead of generating search variants, **break the question into focused sub-questions** and answer each one independently, then combine.

```
Original: "Compare Free and Pro plan user limits and what happens when exceeded"
Decomposed:
  Sub-Q1: "What are the user limits on the Free plan?"
  Sub-Q2: "What are the user limits on the Pro plan?"
  Sub-Q3: "What happens when you exceed the user limit on Free?"
```

**Advantage over multi-query:** Each sub-question gets a **dedicated answer**, so the final response is structured and directly addresses each part. Better for "compare X and Y" questions.

In [ ]:
DECOMPOSE_SYSTEM = """You are a question analysis assistant. Your job is to break down
a complex question into 2-4 focused, self-contained sub-questions that together
cover all aspects of the original question.

Return ONLY a JSON array of sub-questions:
[
  {"sub_question": "...", "aspect": "..."},
  ...
]

Each sub-question should be answerable independently from documentation."""

SUB_ANSWER_SYSTEM = """You are a helpful NovaDash support assistant. Answer the user's
question using ONLY the provided documentation context. Be specific and concrete —
cite exact limits, numbers, and policies from the docs. If the docs don't have the
answer, say so clearly."""

SYNTHESIZE_SYSTEM = """You are a helpful NovaDash support assistant. You have received
answers to several sub-questions that together address a user's complex question.
Synthesize these answers into a single, well-structured response.

Use markdown headers to organize the answer. Be concise — don't repeat information."""


def decompose_question(question: str) -> list:
    """Break a complex question into focused sub-questions."""
    response = ask_claude(
        system=DECOMPOSE_SYSTEM,
        user=f"Complex question: {question}",
        model="claude-haiku-4-5-20251001",
    )

    try:
        clean = re.sub(r"```json\s*|\s*```", "", response).strip()
        items = json.loads(clean)
        if isinstance(items, list):
            return items
    except (json.JSONDecodeError, ValueError):
        pass

    # Fallback decomposition
    return [
        {"sub_question": "What are the user and project limits on the Free plan?", "aspect": "Free plan limits"},
        {"sub_question": "What are the user and project limits on the Pro plan?", "aspect": "Pro plan limits"},
        {"sub_question": "What happens when you exceed plan limits in NovaDash?", "aspect": "Exceeding limits"},
    ]


def answer_sub_question(sub_question: str, top_k: int = 4) -> dict:
    """Retrieve context and generate an answer for a single sub-question."""
    hits = hybrid_search(sub_question, top_k=top_k)
    context = "\n\n---\n\n".join([
        f"Source: {h['source']} | Section: {h['section']}\n{h['text']}"
        for h in hits
    ])
    answer = ask_claude(
        system=SUB_ANSWER_SYSTEM,
        user=f"Documentation context:\n{context}\n\nQuestion: {sub_question}",
        model="claude-haiku-4-5-20251001",
    )
    return {
        "sub_question": sub_question,
        "answer": answer,
        "sources": [f"{h['source']} > {h['section']}" for h in hits[:3]],
    }


print("=" * 70)
print("  Approach 2: Query Decomposition")
print("=" * 70)
print(f"\nOriginal Question:\n  '{COMPLEX_QUERY}'\n")

# Step 1: Decompose
print("Step 1: Decomposing into sub-questions...")
sub_questions = decompose_question(COMPLEX_QUERY)
print(f"\nDecomposed into {len(sub_questions)} sub-questions:")
for i, sq in enumerate(sub_questions, 1):
    aspect = sq.get("aspect", "")
    q = sq.get("sub_question", str(sq))
    print(f"  [{i}] ({aspect}): {q}")

In [ ]:
# Step 2: Answer each sub-question independently
print("\nStep 2: Answering each sub-question independently...")
sub_answers = []

for i, sq in enumerate(sub_questions, 1):
    q = sq.get("sub_question", str(sq))
    aspect = sq.get("aspect", f"Part {i}")

    print(f"\n  Sub-question {i} ({aspect}): {q}")
    result = answer_sub_question(q)

    print(f"  Sources used: {', '.join(result['sources'][:2])}")
    print(f"  Answer snippet: {result['answer'][:150].strip()}...")
    sub_answers.append(result)

# Step 3: Synthesize
print("\n\nStep 3: Synthesizing sub-answers into final response...")

synthesis_context = "\n\n---\n\n".join([
    f"Sub-question {i + 1}: {sa['sub_question']}\n\nAnswer: {sa['answer']}"
    for i, sa in enumerate(sub_answers)
])

final_answer = ask_claude(
    system=SYNTHESIZE_SYSTEM,
    user=f"""Original question: {COMPLEX_QUERY}

Sub-question answers:
{synthesis_context}

Synthesize these into a comprehensive, well-organized answer.""",
    model="claude-haiku-4-5-20251001",
)

print("\n" + "=" * 70)
print("  Decomposition Final Answer")
print("=" * 70)
print(final_answer)

## Approach 3: Agentic ReAct Loop

Give Claude a `search` tool and let it **decide**:
1. What to search for
2. Whether it has enough information
3. Whether it needs to search again

**The ReAct pattern:**
```
Thought → Action (search) → Observation (results) → Thought → ... → Final Answer
```

```
Iteration 1: Claude searches "Free plan limits" → reads results
             Thinks: "I have user limits but need project limits"
Iteration 2: Claude searches "project limits by plan" → reads results  
             Thinks: "I need to know what happens when limits are exceeded"
Iteration 3: Claude searches "exceeding plan limits" → reads results
             Thinks: "I now have everything I need"
Final: Claude synthesizes all gathered context into a comprehensive answer
```

This is why **agentic RAG was the first approach that correctly answered our hardest healthcare query** — it adapts its search strategy based on what it finds.

In [ ]:
import anthropic
from utils import hybrid_search

# Define the search tool that Claude can call
SEARCH_TOOL = {
    "name": "search_docs",
    "description": (
        "Search the NovaDash documentation for relevant information. "
        "Use this to find specific facts, policies, limits, and procedures. "
        "You can call this multiple times with different queries to gather "
        "information from different parts of the documentation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query — be specific for better results. "
                               "Example: 'Free plan project limit' is better than 'plans'.",
            }
        },
        "required": ["query"],
    },
}

AGENT_SYSTEM = """You are a helpful NovaDash support assistant with access to the
NovaDash documentation via a search tool.

When answering questions:
1. Think about what information you need
2. Use the search_docs tool to find relevant documentation
3. You can search multiple times with different queries to gather all needed information
4. Once you have enough information, provide a comprehensive, accurate answer
5. Structure your answer clearly with headers when answering multi-part questions
6. Only state facts that are supported by the documentation you've retrieved

IMPORTANT: Prefer multiple specific searches over one broad search. If a question
asks about multiple topics (e.g., user limits AND project limits), search for each
topic separately."""


def run_search(query: str) -> str:
    """Execute a search and return formatted results as a string for the agent."""
    hits = hybrid_search(query, top_k=5)
    if not hits:
        return "No relevant documentation found for this query."

    results = []
    for i, hit in enumerate(hits[:4], 1):
        results.append(
            f"[{i}] Source: {hit['source']} | Section: {hit['section']}\n"
            f"{hit['text'][:600]}"
        )
    return "\n\n---\n\n".join(results)


print("ReAct agent setup complete.")
print("Agent has access to search_docs tool — it will call it multiple times.")

In [ ]:
def run_react_agent(question: str, max_iterations: int = 5) -> dict:
    """
    Run the ReAct agent loop.
    Claude decides what to search for, reads results, searches again if needed.
    Returns the final answer and a trace of all search iterations.
    """
    api_key = keyring.get_password("Anthropic", "API_KEY")
    client = anthropic.Anthropic(api_key=api_key.strip())

    messages = [{"role": "user", "content": question}]
    iterations = []
    total_searches = 0

    print(f"Starting ReAct agent loop (max {max_iterations} iterations)...")
    print("-" * 60)

    for iteration in range(max_iterations):
        print(f"\n[Iteration {iteration + 1}]")

        # Ask Claude with the search tool available
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=2048,
            system=AGENT_SYSTEM,
            tools=[SEARCH_TOOL],
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            # Claude is done — extract the final answer text
            final_answer = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_answer = block.text
            print(f"Agent finished after {iteration + 1} iteration(s).")
            return {
                "final_answer": final_answer,
                "iterations": iterations,
                "total_searches": total_searches,
            }

        elif response.stop_reason == "tool_use":
            # Claude wants to search — execute each tool call
            tool_uses = [b for b in response.content if b.type == "tool_use"]
            tool_results = []

            for tool_use in tool_uses:
                query = tool_use.input.get("query", "")
                total_searches += 1
                print(f"  Searching: '{query}'")

                results = run_search(query)
                results_preview = results[:300]
                print(f"  Results preview: {results_preview[:120]}...")

                iterations.append({
                    "iteration": iteration + 1,
                    "search_query": query,
                    "results_preview": results_preview,
                })

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": results,
                })

            # Add Claude's response + tool results to message history for next iteration
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})

        else:
            print(f"  Unexpected stop_reason: {response.stop_reason}")
            break

    # Reached max iterations — force final answer
    print(f"\nReached max iterations ({max_iterations}). Extracting final answer...")
    final_response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2048,
        system=AGENT_SYSTEM + "\n\nYou have reached the maximum number of searches. "
               "Synthesize what you know into a final answer now.",
        messages=messages + [{"role": "user", "content": "Please provide your final answer now."}],
    )
    final_text = ""
    for block in final_response.content:
        if hasattr(block, "text"):
            final_text += block.text

    return {
        "final_answer": final_text,
        "iterations": iterations,
        "total_searches": total_searches,
    }


print("=" * 70)
print("  Approach 3: Agentic ReAct Loop")
print("=" * 70)
print(f"\nQuestion: {COMPLEX_QUERY}\n")

# Run the agent
result = run_react_agent(COMPLEX_QUERY)

In [ ]:
# Print the full reasoning trace
print("\n" + "=" * 70)
print("  Agent Reasoning Trace")
print("=" * 70)
print(f"\nTotal searches performed: {result['total_searches']}")
for it in result["iterations"]:
    print(f"\n  Iteration {it['iteration']}: Searched for '{it['search_query']}'")
    print(f"  Results: {it['results_preview'][:120].strip()}...")

# Print the final answer
print("\n\n" + "=" * 70)
print("  Final Answer (Agentic ReAct)")
print("=" * 70)
print(result["final_answer"])

# Summary of what the agent did
print("\n\n" + "=" * 70)
print("  Why Agentic RAG Works for This Question")
print("=" * 70)
print(f"\nThe agent made {result['total_searches']} targeted searches:")
for i, it in enumerate(result['iterations']):
    print(f"  {i+1}. '{it['search_query']}'")
print()
print("Each search was MORE targeted than the last — the agent used what it found")
print("to formulate better follow-up queries. This adaptive strategy is")
print("why agentic RAG succeeds where fixed-pipeline approaches struggle.")

## Comparison: All Three Approaches

| Approach | Queries | API Calls | Latency | Quality | Best for |
|---|---|---|---|---|---|
| Single query | 1 | 1 | ~1s | Poor | Simple factual Q&A |
| Multi-query | 4 | 1+ | ~2s | Good | Broad coverage |
| Decomposition | 2-3 | 3-4 | ~4s | Very good | Compare/contrast |
| Agentic ReAct | 2-5 | 5-10 | ~8-15s | Excellent | Anything complex |

For a support chatbot: single-query is fine for most questions.
For a research assistant: agentic ReAct is worth the latency.

**The pivotal insight from our healthcare project:**
> Agentic RAG was the FIRST approach that correctly answered our hardest test query end-to-end. Not because it was clever — because it **adapted**. Fixed-pipeline approaches commit to a search strategy upfront. Agents revise their strategy based on what they find.

In [ ]:
# Final comparison summary
print("Approach Comparison Summary")
print("=" * 70)
print()
approaches = [
    ("Single-query RAG",    "1 search",                "misses critical sections"),
    ("Multi-query",         "4 predefined searches",   "more coverage but not adaptive"),
    ("Decomposition",       "2-3 predefined searches", "structured but not adaptive"),
    ("Agentic ReAct",       f"{result['total_searches']} adaptive searches", "self-correcting, best quality"),
]
for name, searches, note in approaches:
    print(f"  {name:<22} {searches:<28} {note}")

print()
print(f"Agentic cost: {result['total_searches'] + 1} API calls vs 2 for single-query.")
print("For multi-faceted questions: agentic RAG is clearly superior.")

## Summary

For questions that span multiple topics, a single-pass RAG pipeline will always miss part of the answer.

**Multi-query expansion** works when you need broad coverage across a topic. **Query decomposition** works well for compare/contrast questions where each part needs a dedicated answer. **Agentic ReAct** is the most powerful — it adapts based on what it finds — and was the only approach that correctly answered our hardest real-world query.

Start simple. Upgrade to agentic only when query complexity demands it — the latency and cost are real.

---

**Next:** [Chapter 5 — It Hallucinates](../05-it-hallucinates/notebook.ipynb)

Even perfect retrieval can produce hallucinated answers. The bot invents API endpoints, function names, and database columns that don't exist. We'll build a multi-layer defense.